# MUSS — Google Colab Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amazon-science/muss/blob/main/muss_colab_demo.ipynb)

**MUSS: Multilevel Subset Selection for Relevance and Diversity** &nbsp;·&nbsp; [Paper (arXiv:2503.11126)](https://arxiv.org/pdf/2503.11126) &nbsp;·&nbsp; [Vu Nguyen](https://ntienvu.github.io), Andrey Kan (Amazon)

This notebook runs end-to-end on a free Colab CPU instance. It:

1. clones the MUSS repository and installs dependencies,
2. generates a synthetic corpus (N = 100k items, d = 50),
3. selects `k = 50` diverse, high-quality items with MUSS, and
4. compares MUSS against random selection — numerically and visually.

> Just press **Runtime → Run all**.

## 1. Setup — clone the repo and install dependencies

On Colab this clones `amazon-science/muss` into `/content/muss` and installs the runtime packages MUSS needs. Locally (the repo already present), the clone is skipped automatically.

In [ ]:
import os, sys

# Clone the repository when running on a fresh Colab machine.
if not os.path.isdir('muss'):
    !git clone --depth 1 https://github.com/amazon-science/muss.git

# Minimal dependencies for this demo (the full requirements.txt also covers the
# RAG / candidate-retrieval experiments, which this notebook does not run).
!pip install -q faiss-cpu numpy scikit-learn pandas matplotlib

In [ ]:
# The MUSS package is imported as `muss.src...`, so the *parent* of the `muss/`
# folder must be on the path. On Colab that is /content (the default cwd).
from pathlib import Path

project_root = str(Path().resolve())
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print('Project root:', project_root)

In [ ]:
import random
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import pairwise_distances

from muss.src.methods.muss import muss
print('MUSS imported successfully.')

## 2. Generate synthetic data

Each item has a `d`-dimensional embedding `x` and a scalar quality score `q`.

In [ ]:
def sample_data(N, d):
    np.random.seed(42)
    x = np.random.rand(N, d)   # item embeddings
    q = np.random.rand(N)      # quality scores
    return x, q

N = 100_000   # number of items in the corpus
k = 50        # number of items to select

x, q = sample_data(N, d=50)
print('number of data points  N =', N)
print('number of selected     k =', k)
print('embeddings x:', x.shape, '| quality q:', q.shape)

## 3. Run MUSS

MUSS runs in three stages: **partition** the corpus, **select clusters**, then **select within** each chosen cluster. The `diagnostics` dict reports per-stage timings.

| Parameter | Meaning |
|---|---|
| `k` | items to select |
| `m` | partitions selected in stage 2 |
| `n_partitions` | partitions created in stage 1 (must be > `m`) |

In [ ]:
selected_idx, diagnostics = muss(x, q, k, m=100, n_partitions=200)

print('selected indices:', selected_idx)
print('\ndiagnostics:', diagnostics)

## 4. Compare against a random baseline

A good selection has **high average quality** *and* **high average pairwise distance** (diversity).

In [ ]:
random.seed(0)
selected_random_idx = random.sample(range(N), k)

def avg_quality(idx):
    return float(np.mean(q[idx]))

def avg_pairwise_dist(idx):
    d = pairwise_distances(x[idx, :], x[idx, :], metric='euclidean')
    return float(np.mean(d))

summary = pd.DataFrame(
    {
        'Method':            ['MUSS', 'Random'],
        'Avg Quality':       [round(avg_quality(selected_idx), 4),       round(avg_quality(selected_random_idx), 4)],
        'Avg Pairwise Dist': [round(avg_pairwise_dist(selected_idx), 4), round(avg_pairwise_dist(selected_random_idx), 4)],
    }
).set_index('Method')
summary

## 5. Visualize the selection

We project the high-dimensional embeddings to 2D with PCA. The left panel shows where MUSS places its picks (colored by quality); the right panel compares the quality / diversity trade-off.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# 2D projection of a background sample plus the selected points.
rng = np.random.default_rng(0)
bg = rng.choice(N, size=3000, replace=False)
pca = PCA(n_components=2).fit(x[bg])
bg_2d     = pca.transform(x[bg])
muss_2d   = pca.transform(x[selected_idx])
random_2d = pca.transform(x[selected_random_idx])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: spatial layout of MUSS picks over the corpus.
ax1.scatter(bg_2d[:, 0], bg_2d[:, 1], s=6, c='lightgray', alpha=0.5, label='corpus (sample)')
sc = ax1.scatter(muss_2d[:, 0], muss_2d[:, 1], c=q[selected_idx], cmap='viridis',
                 s=70, edgecolors='black', linewidths=0.6, label='MUSS selected')
fig.colorbar(sc, ax=ax1, label='quality')
ax1.set_title('MUSS selection spread across the corpus (PCA)')
ax1.set_xlabel('PC 1'); ax1.set_ylabel('PC 2'); ax1.legend(loc='upper right')

# Right: quality vs diversity trade-off.
methods = ['MUSS', 'Random']
quals   = [avg_quality(selected_idx),       avg_quality(selected_random_idx)]
divs    = [avg_pairwise_dist(selected_idx), avg_pairwise_dist(selected_random_idx)]
colors  = ['#2ca02c', '#d62728']
ax2.scatter(divs, quals, c=colors, s=160, edgecolors='black')
for m_, d_, qa_ in zip(methods, divs, quals):
    ax2.annotate(m_, (d_, qa_), textcoords='offset points', xytext=(8, 8), fontsize=12)
ax2.set_xlabel('Avg pairwise distance  (diversity \u2192)')
ax2.set_ylabel('Avg quality  (\u2191)')
ax2.set_title('Quality vs. diversity (top-right is better)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Next steps

- Try different trade-offs: pass `lamb` (item-level) and `lamb_c` (cluster-level) to `muss(...)` — `0` favors diversity, `1` favors quality.
- Swap in your own `x` (embeddings) and `q` (quality scores).
- See the [README](https://github.com/amazon-science/muss) for the RAG and candidate-retrieval experiments and baselines.

```bibtex
@inproceedings{nguyen2026muss,
  title     = {MUSS: Multilevel Subset Selection for Relevance and Diversity},
  author    = {Nguyen, Vu and Kan, Andrey},
  booktitle = {Uncertainty in Artificial Intelligence (UAI)},
  year      = {2026},
  url       = {https://arxiv.org/pdf/2503.11126}
}
```